In [ ]:
# Installations
!pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-groq transformers sentence-transformers torch rank-bm25 chromadb pymupdf pypdf python-dotenv tqdm numpy pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/

In [ ]:
import os
from dotenv import load_dotenv

# Document Loading
from langchain_community.document_loaders import PyMuPDFLoader

# Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Database
from langchain_chroma import Chroma

# LLM
from langchain_groq import ChatGroq

# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# BM25
from langchain_community.retrievers import BM25Retriever

# Hybrid Retrieval
from langchain_classic.retrievers import EnsembleRetriever

# Cross Encoder
from sentence_transformers import CrossEncoder

from tqdm import tqdm

/tmp/ipykernel_828/1575081467.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [ ]:
load_dotenv()
GROQ_API_KEY = os.environ['GROQ_API_KEY']
if GROQ_API_KEY is None:
  raise ValueError("GROQ_API_KEY is not set")

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [ ]:
DATA_PATH = "/content/drive/MyDrive/RAG_Article/data"

In [ ]:
documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages.")

Loaded 36 pages.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 30


In [ ]:
# ChromaDB Path
CHROMA_DB_DIR = "/content/drive/MyDrive/RAG_Article/chroma_db"

In [ ]:
vector_store = Chroma.from_documents(
  documents=chunks,
  embedding=embedding_model,
  persist_directory=CHROMA_DB_DIR
)

In [ ]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

In [ ]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [ ]:
bm25_retriever.k = 4

In [ ]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[
        dense_retriever,
        bm25_retriever
    ],
    weights=[0.5, 0.5]
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer is not contained in the context, say:
"I couldn't find that information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [ ]:
def rerank_documents(query, documents, top_k=5):
    # Create (query, document) pairs
    sentence_pairs = [
        (query, doc.page_content)
        for doc in documents
    ]

    # Predict relevance scores
    scores = reranker.predict(sentence_pairs)

    # Combine documents with scores
    ranked_docs = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Return top_k
    return ranked_docs[:top_k]

In [ ]:
query = "What are the different array operations?"

candidate_docs = ensemble_retriever.invoke(query)

In [ ]:
reranked_docs = rerank_documents(
    query,
    candidate_docs,
    top_k=2
)

final_docs = [doc for doc, score in reranked_docs]

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
context = format_docs(final_docs)

formatted_prompt = prompt.format(
    context=context,
    question=query
)

In [ ]:
response = llm.invoke(formatted_prompt)
print(response.content)

The different array operations are:

- Searching an element  
- Insertion of a new element  
- Deletion of a required element  
- Modification of an element  
- Merging of arrays
